In [ ]:
# Librerías
import numpy as np
import pandas as pd
import shap
import seaborn as sns

from google.colab import drive

import matplotlib.pyplot as plt
from pandas.plotting import scatter_matrix
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# Cargamos el dataset
drive.mount("/content/drive")

df = pd.read_csv("/content/drive/MyDrive/ML/temps.csv")  # Modificar según ruta donde se almacene

print(df.head()) # Los datos cargaron correctamente

In [ ]:
df.info()

In [ ]:
# Mostramos un grafico de dispersión
cols = ["actual", "average", "temp_1", "temp_2", "forecast_noaa", "forecast_acc", "forecast_under", "friend"]
sns.pairplot(df[cols], diag_kind="kde", plot_kws={"alpha": 0.5})
plt.show()

Observamos cierta tendencia lineal en la mayoria de variables.

In [ ]:
# Mostramos un boxplot
df[cols].plot.box(figsize=(10,4), ylabel="grados")
plt.show()

No hay prácticamente outliers y presentan escalas similares, y las variables actual, temp 1 y 2 y friend tienen cierta variabilidad.


In [ ]:
# Convertimos en variables dummies los dias de la semana
df = pd.get_dummies(df, columns=["week"])
df

In [ ]:
# Separamos el data set en datos en entrenamiento y prueba.
X = df.drop("actual", axis=1)
y = df["actual"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

In [ ]:
# Creamos el modelo y predecimos.
modelo = RandomForestRegressor()

param_grid = {
    "n_estimators": [25, 50, 100],
    "max_depth": [None, 20, 50],
    "min_samples_split": [2, 5, 7],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt"],
    }

grid = GridSearchCV(modelo, param_grid, cv=5)
grid.fit(X_train, y_train)

In [ ]:
#Evaluamos el model
print("Mejores parámetros:", grid.best_params_)
print("Mejor precisión (CV):", grid.best_score_)

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)


mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²:", round(r2, 4))

El modelo explica una varianza del 81% e la temperatura.

In [ ]:
# Calculamos la importancia
importancias = pd.Series(best_model.feature_importances_, index=X.columns)
importancias = importancias.sort_values(ascending=False)

print(importancias)

Las variables con más peso son temp_1, average, forecast_acc     

In [ ]:
importancias.plot(kind="bar")
plt.show()

El algoritmo de árbol de decisión individual puede ser limitado, ya que tiende a memorizar los datos de entrenamiento, lo que genera overfitting. Random Forest, al combinar múltiples árboles, elimina este problema; esto sería por la aleatoriedad del entrenamiento, al entrenarse con diferentes muestras. Por otro lado, Random Forest pierde interpretabilidad al no generarse un árbol final, aunque se pueden usar técnicas para mejorar esta, como feature of importance.


In [ ]:
# SHAP
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test)

Para finalizar el ejercicio, realizamos SHAP, donde vemos, como era de esperar, que las temperaturas más altas hacen que el target suba y las más bajas que baje.

